# Imports

In [5]:
import pandas as pd
from retriever import retrieve
from tqdm import tqdm

# TeleQnA

In [ ]:
df = pd.read_json(r"E:\Vignesh\VIT\Project 1\TelecomRAG\Datasets\Telecom\teleqna_3gpp_questions.json")
df.head()

,question_id,question,option 1,option 2,option 3,option 4,category
0,question 0,What is the purpose of the Nmfaf_3daDataManage...,To configure the MFAF to map data or analytics...,To configure the MFAF to stop mapping data or ...,To supply data or analytics from the MFAF to n...,To fetch data or analytics from the MFAF based...,Standards specifications
1,question 4,How does a supporting UE attach to the same co...,It requests the core network node to remember ...,It uses information stored in the UE when it w...,It relies on the SIM/USIM card for information.,It performs a fresh attach procedure.,Standards specifications
2,question 11,When can the setting of the Privacy exception ...,Never,Only during emergency services,Anytime,Only with operator permission,Standards specifications
3,question 12,What does the NEF notify to the AF after deter...,AF Identifier,EAS address information,DNN,DNAI(s) or the updated DNAI information,Standards specifications
4,question 14,"In online charging, how are chargeable events ...",By the CTF,By the OCF,By the CGF,By the CHF,Standards specifications


In [8]:
result=df.sample(10)
preds=[]
for row_num,row in tqdm(result.iterrows(),total=len(result)):
    question=row['question']
    options=[row[f"option {i}"] for i in range(1,5)]
    answer=(retrieve(question,options))
    #print(row)
    #print("Question:",question)
    #print("Options:",row['choices'])
    #print("Answer:",answer)
    preds.append(answer)
print(preds[:20])

 30%|███       | 3/10 [00:43<02:04, 17.76s/it]

Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m23jtgd9efq865j3ncnctqa1` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 2903, Requested 5189. Please try again in 690ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


100%|██████████| 10/10 [04:34<00:00, 27.45s/it]

['2', '3', '', '3', '3', '2', '4', '4', '3', '3']


In [10]:
result['predictions']=preds
print(result.head())
result.to_csv('teleqna_results.csv')
#print(df.iloc[168]['question'])

       question_id                                           question  \
822  question 5664  What is the purpose of the Multi-device and Mu...   
929  question 6212  What is the duration called that the UE waits ...   
459  question 3273  What services can modify the originating netwo...   
158  question 1065  In SNPN access operation mode over 3GPP access...   
620  question 4308  How is the number of logical slots for a reser...   

                                              option 1  \
822  To allow users to use any public user identiti...   
929                                              Cycle   
459                   IMS Multimedia Telephony service   
158                   Lists of forbidden access points   
620  By multiplying the resource pool size by the s...   

                                              option 2  \
822  To support synchronization of communication lo...   
929                                        Active-time   
459                      OIP service a

In [19]:
ans_df = pd.read_json(r"E:\Vignesh\VIT\Project 1\TelecomRAG\Datasets\Telecom\teleqna_3gpp_answers.json")
ans_df.head()

,question_id,answer,explanation,category
0,question 0,2,The Nmfaf_3daDataManagement_Deconfigure servic...,Standards specifications
1,question 4,2,A supporting UE in a shared network attaches t...,Standards specifications
2,question 11,3,The setting of the Privacy exception list can ...,Standards specifications
3,question 12,4,"After determining the suitable DNAI(s), NEF no...",Standards specifications
4,question 14,2,The CTF transforms each chargeable event into ...,Standards specifications


In [ ]:
eval_df=pd.merge(result,ans_df,how='inner',on='question_id')[['question_id', 'question', 'option 1', 'option 2', 'option 3','option 4', 'predictions','answer']]
eval_df

,question_id,question,option 1,option 2,option 3,option 4,predictions,answer
0,question 5664,What is the purpose of the Multi-device and Mu...,To allow users to use any public user identiti...,To support synchronization of communication lo...,To enable the addition and deletion of identit...,To provide information about potential risks a...,2,1
1,question 6212,What is the duration called that the UE waits ...,Cycle,Active-time,On-duration,Inactivity-timer,3,3
2,question 3273,What services can modify the originating netwo...,IMS Multimedia Telephony service,OIP service and eCNAM service,Additional timing information services,User configuration services,1,2
3,question 1065,In SNPN access operation mode over 3GPP access...,Lists of forbidden access points,Lists of forbidden tracking areas,Lists of selected SNPNs,Lists of PLMN subscriptions,3,2
4,question 4308,How is the number of logical slots for a reser...,By multiplying the resource pool size by the s...,By counting the number of slots in a resource ...,By converting the resource reservation period ...,By dividing the resource reservation interval ...,3,3
5,question 405,What are the enhanced RRM requirements in DRX ...,Reduced cell identification delay and measurem...,"Reduced cell detection delay, measurement peri...","Reduced cell identification delay, measurement...",Reduced cell detection delay and measurement p...,2,1
6,question 2190,What is the primary purpose of MPS for MMTEL v...,To provide priority access to enterprise netwo...,To provide priority video communication sessio...,To provide priority data transport service in ...,To provide priority voice communication sessio...,4,4
7,question 3938,What does AMF Load Re-Balancing functionality ...,Stopping redirection of UEs served by one of i...,Moving UEs from CM-IDLE state to CM-CONNECTED ...,Increasing the weight factor of an AMF after i...,Moving UEs registered on an AMF to another AMF...,4,4
8,question 1463,Which specification defines the Nchf_SpendingL...,TS 23.203,TS 32.27x,TS 23.502,TS 23.272,3,3
9,question 3099,Which security procedure is used for RRC and U...,AMF SMC procedure,NAS SMC procedure,AS SMC procedure,UE SMC procedure,3,3


In [24]:
eval_df['answer']=eval_df['answer'].astype(int)
eval_df['predictions']=eval_df['predictions'].astype(int)

In [27]:
len(eval_df[eval_df['answer']==eval_df['predictions']])*100/len(eval_df)

60.0